In [15]:
import pymc as pm
import numpy as np
import arviz as az
import matplotlib.pyplot as plt
import pandas as pd

# Observed data
#trust_levels = np.array([23.0, 45.0, 67.0, 89.0, 34.0, 56.0, 78.0, 12.0, 90.0, 22.0]) / 100
#decisions = np.array([0, 0, 1, 1, 0, 0, 1, 0, 1, 0])


# Load the CSV file
file_path = '/Users/apple/Documents/Fall_2024_Research/1009_event_hardcoded_07_19_2024.csv'  
data = pd.read_csv(file_path)
data = data.head(20)
# Extract column
trust_levels = data['trust_level'].values / 100  # Scale by dividing by 100
decisions = data['move_approved'].values
current_health = data['current_health'].values
proposed_damage = data['proposed_damage'].values

coin_collected = data['coin_collected'].astype(float).values  # Assuming 1 for True, 0 for False

#length
N = len(trust_levels)

#Remaining health after damage
health_margin = current_health - proposed_damage

#model
with pm.Model() as model:
    # Define data variables
    trust_levels_data = pm.Data('trust_levels', trust_levels)
    current_health_data = pm.Data('current_health', current_health)
    proposed_damage_data = pm.Data('proposed_damage', proposed_damage)
    coin_collected_data = pm.Data('coin_collected', coin_collected)
    decisions_data = pm.Data('decisions', decisions)


    # Remaining health after damage
    health_margin = current_health_data - proposed_damage_data

    sigma_theta = 0.05  # Standard deviation for theta evolution

    # Initial threshold prior
    theta_0 = pm.Beta('theta_0', alpha=1, beta=1)

    # Develop a theta list
    theta_list = [theta_0]
    for t in range(1, N):
        theta_proposed = pm.Normal(f'theta_proposed_{t}', mu=theta_list[t-1], sigma=sigma_theta)
        theta_t = pm.Deterministic(f'theta_{t}', pm.math.sigmoid(theta_proposed))
        theta_list.append(theta_t)

    theta = pm.Deterministic('theta', pm.math.stack(theta_list))

    # Define the probabilities
    trust_condition_prob = pm.math.sigmoid((trust_levels_data - theta) * 10)
    beta_health = pm.Normal('beta_health', mu=0, sigma=1)
    health_condition_prob = pm.math.sigmoid(beta_health * health_margin)
    beta_coin = pm.Normal('beta_coin', mu=1, sigma=1)
    beta_damage_with_coin = pm.Normal('beta_damage_with_coin', mu=0.5, sigma=0.5)
    adjusted_damage = pm.math.switch(coin_collected_data, proposed_damage_data * beta_damage_with_coin, proposed_damage_data)
    reward_condition_prob = pm.math.sigmoid(beta_coin * coin_collected_data - adjusted_damage)
    w_trust = pm.Normal('w_trust', mu=0.5, sigma=0.1)
    w_health = pm.Normal('w_health', mu=0.3, sigma=0.1)
    w_reward = pm.Normal('w_reward', mu=0.2, sigma=0.1)

    decision_prob = pm.Deterministic(
        'decision_prob', 
        w_trust * trust_condition_prob + w_health * health_condition_prob + w_reward * reward_condition_prob
    )

    observed = pm.Bernoulli('observed', p=decision_prob, observed=decisions_data)

    # Sample from the posterior
    trace = pm.sample(draws=2000, tune=1000, target_accept=0.9, return_inferencedata=True)


# Update model with new data and generate predictions
with model:
    pm.set_data({
        'trust_levels': trust_levels,
        'current_health': current_health,
        'proposed_damage': proposed_damage,
        'coin_collected': coin_collected
    })

    ppc = pm.sample_posterior_predictive(trace, var_names=['observed'])

# Analyze predictions
predicted_decisions = ppc.posterior_predictive['observed']
print(predicted_decisions)


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [theta_0, theta_proposed_1, theta_proposed_2, theta_proposed_3, theta_proposed_4, theta_proposed_5, theta_proposed_6, theta_proposed_7, theta_proposed_8, theta_proposed_9, theta_proposed_10, theta_proposed_11, theta_proposed_12, theta_proposed_13, theta_proposed_14, theta_proposed_15, theta_proposed_16, theta_proposed_17, theta_proposed_18, theta_proposed_19, beta_health, beta_coin, beta_damage_with_coin, w_trust, w_health, w_reward]


/Users/apple/opt/anaconda3/envs/pymc_env/lib/python3.12/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 157 seconds.
There were 7448 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Sampling: [observed]


/Users/apple/opt/anaconda3/envs/pymc_env/lib/python3.12/site-packages/rich/live.py:231: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

<xarray.DataArray 'observed' (chain: 4, draw: 2000, observed_dim_2: 20)> Size: 1MB
array([[[1, 0, 0, ..., 1, 0, 0],
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        ...,
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        [1, 0, 1, ..., 1, 1, 1]],

       [[1, 0, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 0, 1],
        [1, 0, 1, ..., 0, 0, 1],
        ...,
        [0, 1, 1, ..., 1, 1, 1],
        [0, 1, 1, ..., 1, 0, 0],
        [1, 1, 1, ..., 1, 1, 1]],

       [[1, 1, 0, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 0, ..., 1, 1, 1],
        ...,
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 0, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1]],

       [[1, 0, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 0],
        [1, 0, 1, ..., 0, 1, 1],
        ...,
        [1, 1, 1, ..., 0, 1, 1],
        [1, 1, 1, ..., 1, 1, 0],
        [1, 1, 1, ..., 1, 1, 0]]])
Coordinates:
  * chain           (chain) int64 32B 0 1 2 3
  * dr